# CodonTransformer fine-tuning smoke test on Google Colab

This notebook keeps code in GitHub, downloads the immutable pretrained model from Hugging Face, reads cluster-cleaned training JSONL from Google Drive, and writes checkpoints/logs/configuration back to Drive. Colab local storage is temporary and is not used for persistent checkpoints.

## 1. Check the NVIDIA GPU
Select a GPU runtime before continuing: **Runtime → Change runtime type → GPU**.

In [ ]:
!nvidia-smi
import torch
assert torch.cuda.is_available(), "CUDA GPU is required for this notebook"
print("CUDA device:", torch.cuda.get_device_name(0))

## 2. Configure repository, model, data, and output locations
The repository is private by default. Add a fine-grained read-only token as the Colab secret `GITHUB_TOKEN`; never paste it into this notebook. All paths used later derive from this configuration cell.

In [ ]:
import os
from pathlib import Path

REPO_URL = os.environ.get(
    "GITHUB_REPO_URL",
    "https://github.com/Yuano0o/codontransformer-nb.git",
)
COLAB_WORKSPACE = Path(os.environ.get("COLAB_WORKSPACE", "/content"))
PROJECT_DIR = COLAB_WORKSPACE / "codontransformer-project"
UPSTREAM_REPO_URL = "https://github.com/Adibvafa/CodonTransformer.git"
UPSTREAM_COMMIT = "4a447b01dab860feb81b647ff1ff88ad598517f4"
HF_MODEL_ID = "adibvafa/CodonTransformer"
DRIVE_MOUNT = Path(os.environ.get("COLAB_DRIVE_MOUNT", "/content/drive"))
DRIVE_PROJECT_SUBDIR = Path("MyDrive") / "CodonTransformer"
DRIVE_TRAINING_DATA = Path("data") / "stage2" / "final_csi_cohorts" / "experiments" / "csi_top10_hc" / "train.jsonl"
RUN_NAME = "smoke_test"

assert REPO_URL.startswith("https://github.com/")

## 3. Clone the GitHub project and pinned upstream source

In [ ]:
import subprocess
from google.colab import userdata

try:
    github_token = userdata.get("GITHUB_TOKEN")
except Exception:
    github_token = None
clone_env = os.environ.copy()
if github_token:
    clone_env.update({
        "GIT_CONFIG_COUNT": "1",
        "GIT_CONFIG_KEY_0": "http.extraHeader",
        "GIT_CONFIG_VALUE_0": f"AUTHORIZATION: bearer {github_token}",
    })
if not (PROJECT_DIR / ".git").is_dir():
    subprocess.run(
        ["git", "clone", REPO_URL, str(PROJECT_DIR)],
        check=True,
        env=clone_env,
    )
os.chdir(PROJECT_DIR)

upstream_dir = PROJECT_DIR / "upstream"
if not (upstream_dir / ".git").is_dir():
    subprocess.run(
        ["git", "clone", UPSTREAM_REPO_URL, str(upstream_dir)],
        check=True,
    )
subprocess.run(
    ["git", "-C", str(upstream_dir), "checkout", UPSTREAM_COMMIT],
    check=True,
)
print("Project:", PROJECT_DIR)
print("Upstream commit:", UPSTREAM_COMMIT)

## 4. Install Colab dependencies
The requirements file deliberately leaves Colab's CUDA-enabled PyTorch installation unchanged.

In [ ]:
subprocess.run(
    [os.sys.executable, "-m", "pip", "install", "-r", "requirements-colab.txt"],
    check=True,
)
subprocess.run(
    [os.sys.executable, "-m", "pip", "install", "--no-deps", "-e", "upstream"],
    check=True,
)

## 5. Mount Google Drive and locate cleaned training data
The expected JSONL is the later cluster-aware split in the actual upstream input format (`codons` string plus integer `organism`). Stage-one CSV must not be used directly.

In [ ]:
from google.colab import drive

drive.mount(str(DRIVE_MOUNT))
DRIVE_ROOT = DRIVE_MOUNT / DRIVE_PROJECT_SUBDIR
DATASET_PATH = DRIVE_ROOT / DRIVE_TRAINING_DATA
RUN_DIR = DRIVE_ROOT / "runs" / RUN_NAME
RUN_DIR.mkdir(parents=True, exist_ok=True)
assert DATASET_PATH.is_file(), f"Training JSONL not found: {DATASET_PATH}"
print("Training data:", DATASET_PATH)
print("Persistent run directory:", RUN_DIR)

## 6. Download the pretrained model from Hugging Face
The model is downloaded only to Colab temporary storage. It is never committed to GitHub.

In [ ]:
MODEL_DIR = PROJECT_DIR / "models" / "pretrained"
subprocess.run(
    [
        os.sys.executable,
        "scripts/download_pretrained.py",
        "--repo-id",
        HF_MODEL_ID,
        "--output-dir",
        str(MODEL_DIR),
    ],
    check=True,
)
assert (MODEL_DIR / "model.safetensors").is_file()
print("Temporary model directory:", MODEL_DIR)

## 7. Run the CUDA smoke test
Only two training batches are run. The output directory is on Drive, so checkpoints, Lightning CSV logs, `training.log`, `resolved_config.yaml`, and `runtime.json` persist after Colab disconnects.

In [ ]:
subprocess.run(
    [
        os.sys.executable,
        "scripts/finetune_codontransformer.py",
        "--config",
        "configs/smoke_test.yaml",
        "--model-dir",
        str(MODEL_DIR),
        "--dataset-path",
        str(DATASET_PATH),
        "--output-dir",
        str(RUN_DIR),
    ],
    check=True,
)
CHECKPOINT_PATH = RUN_DIR / "checkpoints" / "last.ckpt"
assert CHECKPOINT_PATH.is_file(), f"Checkpoint not found: {CHECKPOINT_PATH}"
print("Checkpoint saved to Drive:", CHECKPOINT_PATH)

## 8. Reload the checkpoint and verify inference by translation

In [ ]:
VALIDATION_OUTPUT = RUN_DIR / "checkpoint_inference_validation.json"
subprocess.run(
    [
        os.sys.executable,
        "scripts/validate_checkpoint_inference.py",
        "--model-dir",
        str(MODEL_DIR),
        "--checkpoint",
        str(CHECKPOINT_PATH),
        "--output",
        str(VALIDATION_OUTPUT),
        "--device",
        "cuda",
        "--organism",
        "Nicotiana tabacum",
    ],
    check=True,
)
import json
validation = json.loads(VALIDATION_OUTPUT.read_text())
assert validation["translation_verified"] is True
validation

## 9. Inspect persistent Drive outputs
Do not end the session until `last.ckpt` and the validation JSON are visible under the Drive run directory.

In [ ]:
for path in sorted(RUN_DIR.rglob("*")):
    if path.is_file():
        print(path.relative_to(RUN_DIR), path.stat().st_size)